# 13_04_ツールの定義を渡し_呼び出しを実行し_結果を返す

コード：`ch13/blocks/13_04_ツールの定義を渡し_呼び出しを実行し_結果を返す.ipynb`

本文の断片は参照表示し、末尾のセルで前提を補う既存ランナーを実行します。関数・変数・外部処理が省略されているため、断片だけを単独実行しないでください。

モデル・データの取得、または長い計算を含みます。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import os
import pathlib
import sys
import zipfile

CODE = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents,
                        pathlib.Path.cwd() / 'gradient-attention-agent']
             if (p / 'run_all.py').is_file() and (p / 'common/blocks.py').is_file()), None)
if CODE is None:
    from google.colab import files
    uploaded = files.upload()  # gradient-attention-agent-code.zip を選ぶ
    if len(uploaded) != 1:
        raise ValueError('配布ZIPを1つ選んでください')
    dest = pathlib.Path.cwd().resolve()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        for member in z.namelist():
            if not (dest / member).resolve().is_relative_to(dest / 'gradient-attention-agent'):
                raise ValueError('同梱のコード配布ZIPを選んでください')
        z.extractall(dest)
    CODE = dest / 'gradient-attention-agent'
    del uploaded
print('コード一式:', CODE)


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2', 'transformers==5.22.1', 'tiktoken==0.14.0', 'scikit-learn>=1.5,<2', 'pydantic==2.14.2', 'outlines==1.3.3', 'lancedb==0.41.0', 'pyarrow==26.0.1', 'sentence-transformers==6.2.0']], check=True)


## 本文の掲載コード（参照）


```python
# 13 章：ツールの定義を渡し、モデルが出した呼び出しを実行して結果を返す往復を、実モデルで確かめる
import json, re, sys, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ツールの実装（確認用の架空のデータ）
WEATHER = {"東京": "晴れ・18 度", "大阪": "くもり・20 度", "札幌": "雪・-2 度"}
RATE = {("USD", "JPY"): 150.0, ("EUR", "JPY"): 160.0}
def get_weather(city: str) -> str:
    return WEATHER.get(city, "不明")
def convert_currency(amount: float, src: str, dst: str) -> float:
    return round(amount * RATE[(src, dst)], 2)
IMPL = {"get_weather": get_weather, "convert_currency": convert_currency}

# ツールの定義：名前・説明・引数の JSON スキーマ。モデルにはこれだけが渡る
TOOLS = [
    {"type": "function", "function": {
        "name": "get_weather", "description": "都市の今日の天気と気温を返す。",
        "parameters": {"type": "object", "properties": {
            "city": {"type": "string", "description": "都市名（例：東京）"}}, "required": ["city"]}}},
    {"type": "function", "function": {
        "name": "convert_currency", "description": "金額を別の通貨に換算する。",
        "parameters": {"type": "object", "properties": {
            "amount": {"type": "number", "description": "金額"},
            "src": {"type": "string", "description": "元の通貨の 3 文字コード（例：USD）"},
            "dst": {"type": "string", "description": "換算先の通貨の 3 文字コード（例：JPY）"}},
            "required": ["amount", "src", "dst"]}}},
]

name = sys.argv[1] if len(sys.argv) > 1 else "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name).eval()

@torch.no_grad()
def chat(messages):
    ids = tok.apply_chat_template(messages, tools=TOOLS, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    out = model.generate(**ids, max_new_tokens=200, do_sample=False)
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)

def tool_calls(text):                  # Qwen2.5 は <tool_call>{...}</tool_call> の形で呼び出しを書く
    calls, broken = [], 0
    for m in re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", text, re.S):
        try:
            calls.append(json.loads(m))
        except json.JSONDecodeError:
            broken += 1
    return calls, broken

def run(question):                     # 1 往復：呼び出し → 実行 → 結果を返す → 最終回答
    messages = [{"role": "user", "content": question}]
    first = chat(messages)
    calls, broken = tool_calls(first)
    if not calls:
        return {"calls": [], "broken": broken, "final": first}
    messages.append({"role": "assistant", "content": "", "tool_calls": [
        {"type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}} for c in calls]})
    for c in calls:
        try:
            result = IMPL[c["name"]](**c["arguments"])
        except Exception as e:          # 存在しないツール・引数の誤りは、エラーとして返す
            result = f"ERROR: {type(e).__name__}: {e}"
        messages.append({"role": "tool", "content": json.dumps(result, ensure_ascii=False)})
    return {"calls": calls, "broken": broken, "final": chat(messages)}

# 質問と、期待する呼び出し（None はツールが要らない質問）
CASES = [
    ("東京の今日の天気は？", [("get_weather", {"city": "東京"})]),
    ("大阪は今日、何度くらい？", [("get_weather", {"city": "大阪"})]),
    ("札幌の天気を教えて。", [("get_weather", {"city": "札幌"})]),
    ("100 ドルは何円？", [("convert_currency", {"amount": 100, "src": "USD", "dst": "JPY"})]),
    ("25 ユーロを円にすると？", [("convert_currency", {"amount": 25, "src": "EUR", "dst": "JPY"})]),
    ("1,200 USD を JPY に換算して。", [("convert_currency", {"amount": 1200, "src": "USD", "dst": "JPY"})]),
    ("東京と大阪の天気をそれぞれ教えて。", [("get_weather", {"city": "東京"}), ("get_weather", {"city": "大阪"})]),
    ("札幌の天気と、50 ドルが何円かを教えて。", [("get_weather", {"city": "札幌"}), ("convert_currency", {"amount": 50, "src": "USD", "dst": "JPY"})]),
    ("日本の首都はどこ？", None),
    ("「ありがとう」を英語で言うと？", None),
    ("3 と 4 を足すといくつ？", None),
    ("天気予報は、どうやって作られているの？", None),
]
summary = {"ツールの選択が正しい": 0, "引数まで正しい": 0, "JSON が壊れた呼び出し": 0, "要らないのに呼んだ": 0}
raw = []
for q, expected in CASES:
    r = run(q)
    got = [(c.get("name"), c.get("arguments")) for c in r["calls"]]
    summary["JSON が壊れた呼び出し"] += r["broken"]
    if expected is None:
        summary["要らないのに呼んだ"] += bool(got)
        ok_tool = ok_args = not got
    else:
        ok_tool = sorted(n for n, _ in got) == sorted(n for n, _ in expected)
        norm = lambda a: {k: (float(v) if isinstance(v, (int, float)) else v) for k, v in (a or {}).items()}
        ok_args = ok_tool and sorted(map(lambda x: json.dumps(norm(x[1]), sort_keys=True, ensure_ascii=False), got)) == \
                  sorted(map(lambda x: json.dumps(norm(x[1]), sort_keys=True, ensure_ascii=False), expected))
    summary["ツールの選択が正しい"] += ok_tool; summary["引数まで正しい"] += ok_args
    raw.append({"q": q, "expected": expected, "calls": r["calls"], "final": r["final"]})
    print(f"{'○' if ok_args else '×'} {q}  呼び出し: {got}")
print(f"モデル：{name}")
for k, v in summary.items():
    print(f"{k}: {v}" + ("/12" if k in ("ツールの選択が正しい", "引数まで正しい") else ("/4" if k == "要らないのに呼んだ" else "")))
json.dump(raw, open("function_calling_raw.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
```


## 前提を補って実行する

既存ランナーが関連する断片と必要なデータ・スタブを接続します。外部サービスを実行した証明や、掲載用AI応答の完全証跡にはなりません。


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(CODE / "run_all.py"), "--only", "13", "--name=function_calling", "--skip-docker"], cwd=CODE, check=True)
